In [0]:
from pyspark.sql.functions import current_timestamp, input_file_name

# 1. Definir la ruta usando comodines (*) para recorrer año y mes de forma dinámica
# Como usas Unity Catalog y External Locations, la ruta abfss resolverá la autenticación automáticamente
source_path = "abfss://landing@stroutemindeuskadidev.dfs.core.windows.net/weather/air_quality/historical_readings/*/*.json"
delta_path = "abfss://bronze@stroutemindeuskadidev.dfs.core.windows.net/delta_tables/air_quality/data"
deltaTable = "dbw_routemind_euskadi_dev.bronze.air_quality_history"

# 2. Lectura masiva de todos los archivos JSON paginados
# Mantenemos multiline=true para evitar cortes de registros si la API devuelve arrays anidados
df_air_quality_hist_raw = spark.read \
    .option("multiline", "true") \
    .json(source_path)

In [0]:
df_air_quality_hist_bronze = df_air_quality_hist_raw \
    .withColumn("ingestion_timestamp", current_timestamp()) \
    .withColumn("source_file", input_file_name())

In [0]:

df_air_quality_hist_bronze.write.option("path", delta_path).option("mergeSchema", True).saveAsTable(name=deltaTable, format="delta", mode="append")

spark.sql(f"CREATE TABLE IF NOT EXISTS dbw_routemind_euskadi_dev.bronze.air_quality_history USING DELTA LOCATION '{delta_path}'")


In [0]:
%sql
select * from dbw_routemind_euskadi_dev.bronze.air_quality_stations LIMIT 20

In [0]:
%sql
select * from dbw_routemind_euskadi_dev.bronze.cultural_places 
LIMIT 20 